Here’s a beginner‑friendly walk‑through of all the core ideas, step by step, without examples.



### Dense networks vs. CNNs for images

- A **dense (fully connected) network** treats the input as one long list of numbers.  
- For images, this means flattening the 2D pixels into 1D, which throws away the natural 2D structure (which pixels are next to each other).  
- Because of this, a dense network has to separately learn “what a shape looks like” at each position in the image, making it less efficient and less robust to moving objects around.  

A **convolutional neural network (CNN)** keeps and exploits the 2D structure of images, so it can reuse the same visual detector across all positions.



### Translation invariance

- **Translation invariance** means the model gives (roughly) the same prediction even if the object moves to a different location in the image.  
- CNNs achieve this because the same filters (kernels) slide across the entire image: the network does not care where in the image the feature appears, only whether it appears.  
- Pooling (like max pooling) and repeated downsampling further help make the representation less sensitive to small shifts and translations.  

This is why CNNs handle objects at different positions much better than plain dense networks.



### CNNs in Keras: overall structure

In Keras, you build a CNN by stacking layers that operate on 2D image data:

1. **Input (2D image with channels)**  
2. **Convolutional layers** (learn local features with filters/kernels)  
3. **Pooling layers** (shrink spatial size, keep important responses)  
4. Optionally repeat steps 2–3 multiple times  
5. **Flatten layer** (convert feature maps to a 1D vector)  
6. **Dense (fully connected) layers**  
7. **Output layer** (often softmax for classification)  

The key difference from a dense network is that early layers are convolution + pooling on 2D data, not dense on flattened data.



### Sequential API vs. Functional API in Keras

Keras has two main ways to define models:

#### Sequential API

- You **stack layers in order**, like a straight pipeline.  
- You write something like: “start with Input → Conv → Pool → Conv → Pool → Flatten → Dense → Output.”  
- It’s simple and works well when the model goes strictly from layer 1 to layer 2 to layer 3, etc.  

#### Functional API

- You treat layers like functions that take tensors in and return tensors out.  
- You explicitly define how data flows: you call a layer on a previous layer’s output and get a new tensor.  
- This allows more complex topologies (multiple inputs, multiple outputs, branches, merges, skip connections), but can also express simple sequential architectures.  

In this video, both APIs are used to create the *same* CNN; they just provide different coding styles.



### Convolutional layers and their parameters

A **convolutional layer** in Keras (e.g., `Conv2D`) learns a set of small filters that slide over the image and produce feature maps. Key concepts:

#### Number of filters (k1, k2, k3, …)

- The **number of filters** is how many different pattern detectors the layer learns.  
- For example, you might choose 32 filters in the first conv layer, 64 in the second, 128 in the third.  
- As you go deeper, you usually increase the number of filters so the network can represent more complex, high‑level features.  

These counts (32, 64, 128, etc.) are **hyperparameters** you choose; there is no single “correct” value.

#### Kernel size

- The **kernel size** is the height × width of each filter (for example, 3×3).  
- Each filter looks at a small local patch of the input and computes a weighted combination.  

Kernel size, stride, padding, and activation are all part of how the convolutional layer behaves.



### Why parameter counts look the way they do

Each convolutional layer has a certain number of **trainable parameters** (weights + biases). The pattern is:

- For a conv layer with:  
  - `F` filters  
  - each filter of size `kh × kw`  
  - and `C_in` input channels (depth of the input)  
- The number of weights per filter is `kh × kw × C_in`.  
- Each filter also has **1 bias**.  
- So total parameters in the layer = `F × (kh × kw × C_in + 1)`.

Important points:

- **More filters** → more parameters.  
- **More input channels** (deeper input) → more parameters in each filter.  
- **Kernel size** affects how many weights each filter has.  

Max pooling layers and flatten layers have **no trainable parameters** because they only perform fixed operations (downsampling or reshaping), not learned transformations.



### Max pooling layers

- A **max pooling layer** shrinks the spatial size (height and width) of the feature maps.  
- It works by taking non‑overlapping or overlapping blocks (for example, 2×2) and keeping only the maximum value from each block.  
- This does two main things:
  - Reduces computation and memory in deeper layers.  
  - Provides some invariance to small shifts and distortions.  

Max pooling has **no weights**; it simply applies a predefined rule.



### Flatten layer

- The **flatten** layer converts the multi‑dimensional tensor of feature maps (height × width × channels) into a 1D vector.  
- This is needed before feeding into dense layers, which expect a 1D input per example.  
- Flattening does not learn anything; it just reshapes data and has no parameters.



### Model summary in Keras

- The **model summary** is a printed table that shows:
  - Each layer’s name and type.  
  - The output shape of each layer (height, width, channels, etc.).  
  - The number of parameters (weights + biases) for each layer and in total.  
- This is extremely useful for verifying:
  - That shapes are as you expect (no accidental shrinking/expansion).  
  - That layers without trainable parameters (max pooling, flatten) indeed show 0 params.  
  - How large your model is overall (total parameter count).  

In this CNN, the total parameter count (over all conv and dense layers) is a bit over 100k. All of these are trained via gradient descent.



### Training a CNN: same process as dense networks

Conceptually, training a CNN is the same as training any other neural network:

1. **Forward pass**:  
   - Input image goes through conv, activation, pooling, flatten, dense, output layers to produce predictions.  

2. **Loss computation**:  
   - Compare predictions to true labels with a loss function (for classification, usually cross‑entropy).  

3. **Backpropagation**:  
   - Compute gradients of the loss with respect to all weights (filters and dense weights).  

4. **Parameter update**:  
   - Use an optimizer (like SGD or Adam) to adjust the weights in the direction that reduces loss.  

5. **Repeat over many epochs**:  
   - Go through the dataset many times until performance stops improving significantly.

CNNs often need more computation than small dense networks, which is why using a GPU is important for reasonable training times.



### Using GPUs for CNN training

- CNN training involves many convolution operations, which are highly parallel.  
- GPUs are designed for parallel numeric computations, so they speed up convolutions dramatically.  
- When training feels very slow on a CPU, switching the runtime to GPU can reduce training time from many minutes (or hours) to much less.  

In practice, almost all serious CNN training is done on GPUs (or TPUs).



### Performance improvement vs. dense networks

- On the same image task, a well‑designed CNN usually outperforms a plain dense network:
  - It uses spatial structure and weight sharing.  
  - It is more robust to translation and small distortions.  
  - It learns hierarchical visual features (edges → parts → objects).  

So, for image recognition tasks, CNNs are the standard choice over pure dense architectures.
